In [1]:
# import libraries
import json
import itertools

import pandas as pd
import numpy as np
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,average_precision_score,f1_score,
    precision_score,recall_score,classification_report,confusion_matrix
)
RANDOM_STATE=42

### Load feature tables and labels from Notebook 5

In [2]:
X_train=pd.read_csv(r"D:\Olist\train_feature.csv")
X_val=pd.read_csv(r"D:\Olist\validation_feature.csv")
X_test=pd.read_csv(r"D:\Olist\test_feature.csv")

y_train=pd.read_csv(r"D:\Olist\train_labels.csv")["is_late"]
y_val=pd.read_csv(r"D:\Olist\validation_labels.csv")["is_late"]
y_test=pd.read_csv(r"D:\Olist\test_labels.csv")["is_late"]

print(f"Train:      X={X_train.shape},y={y_train.shape}")
print(f"validation: X={X_val.shape},  y={y_val.shape}")
print(f"Test:       X={X_test.shape}, y={y_test.shape}")

print(f"Late rate train: {y_train.mean():.1%}, validation: {y_val.mean():.1%}, test: {y_test.mean():.1%}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Olist\\train_feature.csv'

### Choosing a metric for an imbalanced problem

In [80]:
def evaluate(model,X,y,threshold=0.50):
    y_proba=model.predict_proba(X)[:, 1]
    y_pred=(y_proba >= threshold).astype(int)

    return{
        "roc_auc":roc_auc_score(y, y_proba),
        "pr_auc":average_precision_score(y, y_proba),
        "f1":f1_score(y, y_pred, zero_division=0),
        "precision":precision_score(y, y_pred, zero_division=0),
        "recall":recall_score(y, y_pred, zero_division=0),
        "accuracy":(y_pred == y).mean(),
    }

## Baseline

In [81]:
baseline_model=DummyClassifier(strategy="most_frequent",random_state=RANDOM_STATE)
baseline_model.fit(X_train,y_train)

baseline_metrics=evaluate(baseline_model,X_val,y_val)

print("Baseline model - validation metrics:")
for k,v in baseline_metrics.items():
    print(f"   {k:>10}:   {v:.4f}")

Baseline model - validation metrics:
      roc_auc:   0.5000
       pr_auc:   0.0197
           f1:   0.0000
    precision:   0.0000
       recall:   0.0000
     accuracy:   0.9803


### Logistic Regression model

In [82]:
log_model=LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE)
log_model.fit(X_train,y_train)

log_model_metric=evaluate(log_model, X_val, y_val)

print("Logistic Regression - Validation Metrics:")
for k,v in log_model_metric.items():
    print(f"        {k:>10}:  {v:.4f}")

Logistic Regression - Validation Metrics:
           roc_auc:  0.7516
            pr_auc:  0.0598
                f1:  0.0839
         precision:  0.0449
            recall:  0.6429
          accuracy:  0.7234


### Random Forest Classifier model

In [83]:
rf_model=RandomForestClassifier(
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_model_metrics=evaluate(rf_model, X_val, y_val)

print("Random Forest - Validation Metrics:")
for k, v in rf_model_metrics.items():
    print(f"   {k:>10}:  {v:.4f}")

Random Forest - Validation Metrics:
      roc_auc:  0.7200
       pr_auc:  0.0538
           f1:  0.0342
    precision:  0.1053
       recall:  0.0204
     accuracy:  0.9773


In [84]:
param_grid={
    "n_estimators":[100,300],
    "max_depth":[6,12,None],
    "min_samples_leaf":[1,10]
}

grid_results=[]

for n_estimators, max_depth, min_samples_leaf in itertools.product(
    param_grid["n_estimators"], param_grid["max_depth"], param_grid["min_samples_leaf"]
):
    model=RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    metrics=evaluate(model, X_val, y_val)

    grid_results.append({
        "n_estimators":n_estimators,
        "max_depth":max_depth,
        "min_samples_leaf":min_samples_leaf,
        **metrics
    })

grid_results_df=pd.DataFrame(grid_results).sort_values("pr_auc", ascending=False)
grid_results_df.head(10)

,n_estimators,max_depth,min_samples_leaf,roc_auc,pr_auc,f1,precision,recall,accuracy
5,100,NaN,10,0.745487,0.070429,0.123369,0.080371,0.265306,0.925684
11,300,NaN,10,0.755344,0.069343,0.127711,0.083596,0.270408,0.927192
9,300,12.0,10,0.735722,0.066517,0.109204,0.064457,0.357143,0.885157
8,300,12.0,1,0.731870,0.064699,0.115942,0.069601,0.346939,0.895716
3,100,12.0,10,0.732240,0.064230,0.107897,0.063712,0.352041,0.885257
2,100,12.0,1,0.727578,0.063345,0.121212,0.072581,0.367347,0.895012
4,100,NaN,1,0.719971,0.053763,0.034188,0.105263,0.020408,0.977273
10,300,NaN,1,0.726647,0.053593,0.017857,0.071429,0.010204,0.977876
7,300,6.0,10,0.660994,0.044309,0.092018,0.051617,0.423469,0.835278
0,100,6.0,1,0.664397,0.044059,0.088790,0.049813,0.408163,0.834875


### Pick up the winning hyperparameters and refit the final modef

In [85]:
best_param=grid_results_df.iloc[0][["n_estimators","max_depth","min_samples_leaf"]].to_dict()
best_param["n_estimators"]=int(best_param["n_estimators"])
best_param["min_samples_leaf"]=int(best_param["min_samples_leaf"])
if not pd.isna(best_param["max_depth"]):
    best_param["max_depth"]=int(best_param["max_depth"])
else:
    best_param["max_depth"]=None

print(f"Best hyperparameters: {best_param}")

final_model=RandomForestClassifier(
    **best_param, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
final_model.fit(X_train, y_train)
final_val_metrics=evaluate(final_model, X_val, y_val)

print("Tuned Random Forest - Validation metrics:")
for k,v in final_val_metrics.items():
    print(f"        {k:>10}:  {v:.4f}")

Best hyperparameters: {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 10}
Tuned Random Forest - Validation metrics:
           roc_auc:  0.7455
            pr_auc:  0.0704
                f1:  0.1234
         precision:  0.0804
            recall:  0.2653
          accuracy:  0.9257


### Test the model

In [86]:
test_metrics=evaluate(final_model, X_test, y_test)

print("Tuned Random forest - TEST metrics")
for k,v in test_metrics.items():
    print(f"      {k:>10}: {v:.4f}")
y_test_pred=final_model.predict(X_test)
print("Classification report TEST:")
print(classification_report(y_test, y_test_pred, target_names=["on_time","late"], zero_division=0))

print("Confusion matrix TEST:")
print(confusion_matrix(y_test, y_test_pred))

Tuned Random forest - TEST metrics
         roc_auc: 0.5365
          pr_auc: 0.0853
              f1: 0.0796
       precision: 0.0689
          recall: 0.0943
        accuracy: 0.8187
Classification report TEST:
              precision    recall  f1-score   support

     on_time       0.92      0.88      0.90      9118
        late       0.07      0.09      0.08       827

    accuracy                           0.82      9945
   macro avg       0.49      0.49      0.49      9945
weighted avg       0.84      0.82      0.83      9945

Confusion matrix TEST:
[[8064 1054]
 [ 749   78]]


### Artifact: the trained model and a results summary

In [90]:
results_summary={
    "metric_used_for_selection":"pr_auc (average precision) on the validation split",
    "class_balance":{
        "train_late_rate": float(y_train.mean()),
        "validation_late_rate":float(y_val.mean()),
        "test_late_rate":float(y_test.mean()),
    },
    "baseline_majority_class":baseline_metrics,
    "logistic_ragression_defualt":log_model_metric,
    "random_forest_defualte":rf_model_metrics,
    "random_forets_tuning_leaderbourd":grid_results_df.to_dict(orient="records"),
    "final_model":{
        "type":"RandomForestClassifier",
        "hyperparameter":best_param,
        "validation_metrics":final_val_metrics,
        "test_metrics":test_metrics
    },
}
with open("results_sumarry.json","w") as f:
    json.dump(results_summary, f, indent=2)
joblib.dump(final_model,"final_model.joblib")
print("Artifacts saved: final_model.joblib, results_summary.json")


Artifacts saved: final_model.joblib, results_summary.json
